In [75]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [76]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from src.load_clean_data import load_all_clean_data

dfs = load_all_clean_data()

from src.get_dfs import get_dfs

(circuits,
constructors,
constructor_standings,
drivers,
driver_standings,
qualifying,
races,
results) = get_dfs(dfs)

In [77]:
ml_df = results.copy()

drivers = drivers.rename(columns={
    "name": "driver_name",
    "nationality": "driver_nationality"
})

constructors = constructors.rename(columns={
    "name": "constructor_name",
    "nationality": "constructor_nationality"
})

races = races.rename(columns={
    "date": "race_date",
    "time": "race_time"
})

circuits = circuits.rename(columns={
    "locality": "circuit_locality",
    "country": "circuit_country"
})

qualifying = qualifying.rename(columns={
    "position": "qualifying_position"
})

In [78]:
ml_df.shape

(25939, 13)

In [79]:
ml_df = ml_df.merge(
    drivers,
    on="driver_id",
    how="left"
)

In [80]:
ml_df = ml_df.merge(
    constructors,
    on="constructor_id",
    how="left"
)

display(ml_df.head())
print(ml_df.shape)

,season,round,driver_id,driver_name,constructor_id,grid,position,points,laps,status,time,fastest_lap,fastest_lap_rank,given_name,family_name,dob,driver_nationality,constructor_name,constructor_nationality
0,1950,1,farina,Nino Farina,alfa,1,1,9.0,70,Finished,2:13:23.600,NaN,NaN,Nino,Farina,1906-10-30,Italian,Alfa Romeo,Swiss
1,1950,1,fagioli,Luigi Fagioli,alfa,2,2,6.0,70,Finished,+2.600,NaN,NaN,Luigi,Fagioli,1898-06-09,Italian,Alfa Romeo,Swiss
2,1950,1,reg_parnell,Reg Parnell,alfa,4,3,4.0,70,Finished,+52.000,NaN,NaN,Reg,Parnell,1911-07-02,British,Alfa Romeo,Swiss
3,1950,1,cabantous,Yves Cabantous,lago,6,4,3.0,68,+2 Laps,NaN,NaN,NaN,Yves,Cabantous,1904-10-08,French,Talbot-Lago,French
4,1950,1,rosier,Louis Rosier,lago,9,5,2.0,68,+2 Laps,NaN,NaN,NaN,Louis,Rosier,1905-11-05,French,Talbot-Lago,French


(25939, 19)


In [81]:
ml_df = ml_df.merge(
    races,
    on=["season", "round"],
    how="left"
)

In [82]:
ml_df.shape


(25939, 23)

In [83]:
ml_df = ml_df.merge(
    circuits,
    on="circuit_id",
    how="left"
)

In [84]:
ml_df.shape


(25939, 26)

In [85]:
ml_df = ml_df.merge(
    qualifying,
    on=["season", "round", "driver_id", "constructor_id"],
    how="left"
)

In [86]:
ml_df.shape

(25939, 30)

In [87]:
ml_df.head()

,season,round,driver_id,driver_name,constructor_id,grid,position,points,laps,status,...,circuit_id,race_date,race_time,circuit_name,circuit_locality,circuit_country,qualifying_position,q1,q2,q3
0,1950,1,farina,Nino Farina,alfa,1,1,9.0,70,Finished,...,silverstone,1950-05-13,NaN,Silverstone Circuit,Silverstone,UK,NaN,NaN,NaN,NaN
1,1950,1,fagioli,Luigi Fagioli,alfa,2,2,6.0,70,Finished,...,silverstone,1950-05-13,NaN,Silverstone Circuit,Silverstone,UK,NaN,NaN,NaN,NaN
2,1950,1,reg_parnell,Reg Parnell,alfa,4,3,4.0,70,Finished,...,silverstone,1950-05-13,NaN,Silverstone Circuit,Silverstone,UK,NaN,NaN,NaN,NaN
3,1950,1,cabantous,Yves Cabantous,lago,6,4,3.0,68,+2 Laps,...,silverstone,1950-05-13,NaN,Silverstone Circuit,Silverstone,UK,NaN,NaN,NaN,NaN
4,1950,1,rosier,Louis Rosier,lago,9,5,2.0,68,+2 Laps,...,silverstone,1950-05-13,NaN,Silverstone Circuit,Silverstone,UK,NaN,NaN,NaN,NaN


In [88]:
ml_df["dob"] = pd.to_datetime(ml_df["dob"])
ml_df["race_date"] = pd.to_datetime(ml_df["race_date"])

In [89]:
ml_df["driver_age"] = (
    ml_df["race_date"] - ml_df["dob"]
).dt.days / 365.25

In [90]:
ml_df["podium"] = (
    ml_df["position"] <= 3
).astype(int)

In [91]:
ml_df["win"] = (
    ml_df["position"] == 1
).astype(int)

In [92]:
ml_df["pole_position"] = (
    ml_df["grid"] == 1
).astype(int)

In [93]:
ml_df["front_row"] = (
    ml_df["grid"] <= 2
).astype(int)

In [94]:
# Current season driver points before current race

ml_df["current_season_points"] = (
    ml_df.groupby(["season", "driver_id"])["points"]
    .cumsum()
    - ml_df["points"]
)

In [95]:
# Total driver points per season

season_driver_points = (
    ml_df.groupby(["season", "driver_id"])["points"]
    .sum()
    .reset_index()
)

# Average points across previous seasons

season_driver_points["avg_points_per_season"] = (
    season_driver_points
    .groupby("driver_id")["points"]
    .transform(
        lambda x: (
            x.shift(1)
            .expanding()
            .mean()
        )
    )
)

# Merge back

ml_df = ml_df.merge(
    season_driver_points[
        ["season", "driver_id", "avg_points_per_season"]
    ],
    on=["season", "driver_id"],
    how="left"
)

In [96]:
# Constructor points scored per race

constructor_round_points = (
    ml_df.groupby(
        ["season", "round", "constructor_id"]
    )["points"]
    .sum()
    .reset_index()
)

# Cumulative constructor points BEFORE current race

constructor_round_points[
    "constructor_points_current_season"
] = (
    constructor_round_points
    .groupby(["season", "constructor_id"])["points"]
    .cumsum()
    - constructor_round_points["points"]
)

# Merge back to main dataframe

ml_df = ml_df.merge(
    constructor_round_points[
        [
            "season",
            "round",
            "constructor_id",
            "constructor_points_current_season"
        ]
    ],
    on=["season", "round", "constructor_id"],
    how="left"
)

In [97]:
ml_df.shape

(25939, 38)

In [98]:
ml_df["grid"] = pd.to_numeric(
    ml_df["grid"],
    errors="coerce"
).astype("Int64")

ml_df["qualifying_position"] = pd.to_numeric(
    ml_df["qualifying_position"],
    errors="coerce"
).astype("Int64")

In [99]:
categorical_columns = [
    "driver_id",
    "driver_name",
    "constructor_id",
    "status",
    "given_name",
    "family_name",
    "driver_nationality",
    "constructor_name",
    "constructor_nationality",
    "race_name",
    "circuit_id",
    "circuit_name",
    "circuit_locality",
    "circuit_country"
]

for col in categorical_columns:
    ml_df[col] = ml_df[col].astype("category")

In [102]:
ml_df.to_csv(
    "../data/processed/master_ml_dataset.csv",
    index=False
)

ml_df.info()
ml_df[ml_df["driver_id"] == "bottas"].tail(9)

<class 'pandas.DataFrame'>
RangeIndex: 25939 entries, 0 to 25938
Data columns (total 38 columns):
 #   Column                             Non-Null Count  Dtype         
---  ------                             --------------  -----         
 0   season                             25939 non-null  int64         
 1   round                              25939 non-null  int64         
 2   driver_id                          25939 non-null  category      
 3   driver_name                        25939 non-null  category      
 4   constructor_id                     25939 non-null  category      
 5   grid                               25939 non-null  Int64         
 6   position                           25939 non-null  int64         
 7   points                             25939 non-null  float64       
 8   laps                               25939 non-null  int64         
 9   status                             25939 non-null  category      
 10  time                               8354 non-n

,season,round,driver_id,driver_name,constructor_id,grid,position,points,laps,status,...,q2,q3,driver_age,podium,win,pole_position,front_row,current_season_points,avg_points_per_season,constructor_points_current_season
25290,2024,19,bottas,Valtteri Bottas,sauber,16,17,0.0,55,Lapped,...,NaN,NaN,35.145791,0,0,0,0,0.0,162.545455,0.0
25307,2024,20,bottas,Valtteri Bottas,sauber,15,14,0.0,70,Lapped,...,1:17.817,NaN,35.164956,0,0,0,0,0.0,162.545455,0.0
25326,2024,21,bottas,Valtteri Bottas,sauber,11,13,0.0,69,Finished,...,1:26.472,NaN,35.184120,0,0,0,0,0.0,162.545455,0.0
25351,2024,22,bottas,Valtteri Bottas,sauber,19,18,0.0,49,Lapped,...,NaN,NaN,35.238877,0,0,0,0,0.0,162.545455,0.0
25364,2024,23,bottas,Valtteri Bottas,sauber,13,11,0.0,57,Finished,...,1:21.731,NaN,35.260780,0,0,0,0,0.0,162.545455,0.0
25391,2024,24,bottas,Valtteri Bottas,sauber,9,18,0.0,30,Retired,...,1:23.341,1:23.204,35.279945,0,0,0,0,0.0,162.545455,4.0
25891,2026,1,bottas,Valtteri Bottas,cadillac,19,19,0.0,15,Retired,...,NaN,NaN,36.525667,0,0,0,0,0.0,149.000000,0.0
25907,2026,2,bottas,Valtteri Bottas,cadillac,19,13,0.0,55,Lapped,...,NaN,NaN,36.544832,0,0,0,0,0.0,149.000000,0.0
25935,2026,3,bottas,Valtteri Bottas,cadillac,20,19,0.0,52,Lapped,...,NaN,NaN,36.583162,0,0,0,0,0.0,149.000000,0.0
